This notebook just contains some exploration of the dataset.

In [1]:
import sys

# Necessary to import from src dir
sys.path.append('..')

In [2]:
import matplotlib.pyplot as mlp
import numpy as np
import os
import pandas as pd

from sklearn.preprocessing import StandardScaler

from src.preprocessing import (
    prepare_standardized_datasets,
    standardize_data
)

In [3]:
# Note: notebooks/data is ignored by git, so any files in there will not end up in version control
DATA_DIR = r"./data/"
OUTPUT_DIR = r"./output/"
TEMP_OUTPUT_DIR = r"./temp/"

TRAIN_DATA = os.path.join(DATA_DIR, "train.csv")
TEST_DATA = os.path.join(DATA_DIR, "test.csv")

# Preprocessing

df_train = pd.read_csv(TRAIN_DATA)
df_test = pd.read_csv(TEST_DATA)

label_var = "class4"

# Include only the real-valued mean values, but don't exclude anything further than that yet before exploring the data a bit first
cols_to_include = [feat for feat in df_train.columns.to_list() if (feat.endswith(".mean")) or (feat == label_var)]

numeric_vars = cols_to_include[:]
numeric_vars.remove(label_var)

In [4]:
# Check the Pearson correlation coefficients for the data
# Maybe we can eliminate some redundant variables already at this point

corr = df_train[cols_to_include].corr(numeric_only=True)
corr

,CO2168.mean,CO2336.mean,CO242.mean,CO2504.mean,Glob.mean,H2O168.mean,H2O336.mean,H2O42.mean,H2O504.mean,H2O672.mean,...,SO2168.mean,SWS.mean,T168.mean,T42.mean,T504.mean,T672.mean,T84.mean,UV_A.mean,UV_B.mean,CS.mean
CO2168.mean,1.000000,0.999675,0.993428,0.998567,-0.422862,-0.428546,-0.424609,-0.434061,-0.423042,-0.421750,...,0.313204,-0.144759,-0.565241,-0.563994,-0.568525,-0.567936,-0.564454,-0.434419,-0.434426,-0.106144
CO2336.mean,0.999675,1.000000,0.992175,0.999533,-0.418366,-0.434089,-0.430156,-0.439643,-0.428607,-0.427356,...,0.310494,-0.140993,-0.565057,-0.563768,-0.568681,-0.568213,-0.564217,-0.429937,-0.430238,-0.110800
CO242.mean,0.993428,0.992175,1.000000,0.989651,-0.410648,-0.371312,-0.367937,-0.375666,-0.366559,-0.365313,...,0.316664,-0.157891,-0.523636,-0.523111,-0.526157,-0.525160,-0.523352,-0.416266,-0.406660,-0.082900
CO2504.mean,0.998567,0.999533,0.989651,1.000000,-0.415815,-0.441403,-0.437480,-0.447008,-0.435965,-0.434780,...,0.305035,-0.138026,-0.565800,-0.564382,-0.569942,-0.569724,-0.564860,-0.427277,-0.428167,-0.118085
Glob.mean,-0.422862,-0.418366,-0.410648,-0.415815,1.000000,0.162005,0.156354,0.172892,0.152518,0.151024,...,-0.115704,0.369409,0.594203,0.592415,0.589929,0.589997,0.596149,0.985450,0.935493,0.122259
H2O168.mean,-0.428546,-0.434089,-0.371312,-0.441403,0.162005,1.000000,0.999883,0.999604,0.999704,0.999495,...,-0.242150,-0.239006,0.815735,0.816751,0.820090,0.821259,0.814638,0.252725,0.371318,0.476726
H2O336.mean,-0.424609,-0.430156,-0.367937,-0.437480,0.156354,0.999883,1.000000,0.999138,0.999937,0.999789,...,-0.241652,-0.241590,0.812800,0.813907,0.817181,0.818372,0.811749,0.246731,0.365475,0.478715
H2O42.mean,-0.434061,-0.439643,-0.375666,-0.447008,0.172892,0.999604,0.999138,1.000000,0.998759,0.998440,...,-0.243284,-0.235336,0.820766,0.821548,0.825134,0.826284,0.819552,0.263820,0.382417,0.472611
H2O504.mean,-0.423042,-0.428607,-0.366559,-0.435965,0.152518,0.999704,0.999937,0.998759,1.000000,0.999901,...,-0.241047,-0.242402,0.810517,0.811649,0.814972,0.816216,0.809472,0.242609,0.361413,0.479612
H2O672.mean,-0.421750,-0.427356,-0.365313,-0.434780,0.151024,0.999495,0.999789,0.998440,0.999901,1.000000,...,-0.239400,-0.243683,0.808509,0.809640,0.813044,0.814364,0.807462,0.240687,0.359400,0.480559


In [5]:
def get_n_largest_pearson_coeffs_for_variable(
    corr,
    variable,
    n=10
):
    return np.abs(corr[variable]).nlargest(n)

In [6]:
# We can get the indices of the variables with which the correlation is largest
get_n_largest_pearson_coeffs_for_variable(corr, "CO2168.mean").index

Index(['CO2168.mean', 'CO2336.mean', 'CO2504.mean', 'CO242.mean', 'T504.mean',
       'T672.mean', 'T168.mean', 'T84.mean', 'T42.mean', 'UV_B.mean'],
      dtype='object')

In [7]:
# Or just view the values
get_n_largest_pearson_coeffs_for_variable(corr, "CO2168.mean")

CO2168.mean    1.000000
CO2336.mean    0.999675
CO2504.mean    0.998567
CO242.mean     0.993428
T504.mean      0.568525
T672.mean      0.567936
T168.mean      0.565241
T84.mean       0.564454
T42.mean       0.563994
UV_B.mean      0.434426
Name: CO2168.mean, dtype: float64

Looking at the covariances, we notice very quickly that for $CO_2$ values, it's probably enough to include just one.

In [8]:
# Figure out the columns with the highest overall correlation coefficients
# by calculating the sum of the absolute value of the correlation coefficient table
# and sorting to find those that might be most correlated with others.

n_pearsons_sum_per_col = []

for col in corr.columns:
    n_pearsons_sum_per_col.append(get_n_largest_pearson_coeffs_for_variable(corr, col, len(corr.columns)).sum())

exploratory_table = corr.copy()
exploratory_table["sum_pearson"] = n_pearsons_sum_per_col

exploratory_table["sum_pearson"].nlargest(50)

T84.mean          26.307055
T168.mean         26.270590
T42.mean          26.267059
T504.mean         26.200806
T672.mean         26.170127
UV_B.mean         26.060934
UV_A.mean         26.026486
NET.mean          25.596835
PAR.mean          25.451764
Glob.mean         25.005143
RHIRGA336.mean    24.933839
RHIRGA504.mean    24.930007
RHIRGA672.mean    24.814426
RHIRGA168.mean    24.595503
RHIRGA84.mean     24.303401
RHIRGA42.mean     24.212245
NOx42.mean        22.681271
NOx84.mean        22.673173
NOx168.mean       22.673092
NOx336.mean       22.570084
NOx504.mean       22.499140
NOx672.mean       22.411238
O3672.mean        21.114269
O3504.mean        20.941694
O3168.mean        20.288422
CO2168.mean       20.049462
O384.mean         20.004236
CO2336.mean       19.996899
CO2504.mean       19.946889
O342.mean         19.699757
CO242.mean        19.340127
NO336.mean        19.226499
NO504.mean        19.158918
NO672.mean        19.124453
NO168.mean        19.054896
NO84.mean         18

In [9]:
# If nothing else, we at least may want to keep the variables towards the low end for a longer while

In [10]:
# Prepare standardized datasets

df_train, df_validation, df_test = prepare_standardized_datasets(
    df_train=df_train,
    df_test=df_test,
    data_vars=numeric_vars,
    label_var=label_var,
    label_values=("nonevent", "event"),
    include_validation_split=True,
    validation_split_size=0.2
)


<class 'pandas.core.series.Series'>
(360,)
360


In [11]:
df_train.head()

,class4,CO2168.mean,CO2336.mean,CO242.mean,CO2504.mean,Glob.mean,H2O168.mean,H2O336.mean,H2O42.mean,H2O504.mean,...,SO2168.mean,SWS.mean,T168.mean,T42.mean,T504.mean,T672.mean,T84.mean,UV_A.mean,UV_B.mean,CS.mean
0,nonev,0.556348,0.558433,0.512257,0.564097,-1.323480,-0.543772,-0.533628,-0.560598,-0.522200,...,0.882480,0.434970,-0.883040,-0.874483,-0.882451,-0.886360,-0.885237,-1.293698,-1.179947,0.615379
1,event,-0.727523,-0.722741,-0.711230,-0.731495,1.468801,-0.252151,-0.272936,-0.206721,-0.278886,...,-0.429248,0.466632,1.341433,1.274057,1.349564,1.364608,1.306667,1.456882,1.328232,0.385049
2,nonev,1.692520,1.692304,1.676821,1.695179,-1.431714,-1.187934,-1.188836,-1.186319,-1.190280,...,0.646410,0.151836,-1.831307,-1.814879,-1.851195,-1.868646,-1.819770,-1.478485,-1.298072,-0.389286
3,nonev,-0.421199,-0.403184,-0.456798,-0.390704,0.711573,1.422435,1.441443,1.383440,1.452434,...,-0.481667,0.269989,1.407795,1.434990,1.389909,1.387654,1.426816,0.894577,1.309060,0.797122
4,nonev,0.583889,0.591205,0.531119,0.605068,-0.946114,-0.287740,-0.282375,-0.298645,-0.277423,...,-0.672918,0.231122,-0.525609,-0.523695,-0.532740,-0.537964,-0.528567,-0.823675,-0.749399,-0.853447


In [12]:
df_validation.head()

,class4,CO2168.mean,CO2336.mean,CO242.mean,CO2504.mean,Glob.mean,H2O168.mean,H2O336.mean,H2O42.mean,H2O504.mean,...,SO2168.mean,SWS.mean,T168.mean,T42.mean,T504.mean,T672.mean,T84.mean,UV_A.mean,UV_B.mean,CS.mean
0,nonev,-1.037826,-1.040420,-1.007070,-1.037974,0.592544,1.411653,1.423459,1.404056,1.429111,...,-0.258714,0.481703,1.095870,1.115555,1.109160,1.102935,1.106201,0.696178,1.003049,1.439644
1,event,-0.779498,-0.786251,-0.811032,-0.784304,1.157994,0.350906,0.333899,0.383687,0.323009,...,-0.660936,-0.839446,0.672546,0.675760,0.680523,0.671939,0.674362,1.072439,1.573123,0.439243
2,event,-0.041841,-0.041853,-0.098766,-0.033179,1.086174,-0.654987,-0.648837,-0.665111,-0.645259,...,0.493494,0.719883,0.152867,0.145543,0.125362,0.128137,0.159181,0.885838,0.311952,0.134167
3,nonev,1.152013,1.155270,1.109248,1.163103,-1.396902,-0.778972,-0.771117,-0.786324,-0.769612,...,0.113582,0.168780,-1.146905,-1.140561,-1.147066,-1.152375,-1.144132,-1.429972,-1.282572,-0.963298
4,event,-0.839004,-0.813536,-0.661102,-0.800925,1.512040,0.384595,0.368088,0.433690,0.360008,...,-0.242848,0.358567,1.116718,1.093402,1.112690,1.117660,1.105116,1.600943,2.105601,0.084027


In [13]:
df_test.head()

,CO2168.mean,CO2336.mean,CO242.mean,CO2504.mean,Glob.mean,H2O168.mean,H2O336.mean,H2O42.mean,H2O504.mean,H2O672.mean,...,SO2168.mean,SWS.mean,T168.mean,T42.mean,T504.mean,T672.mean,T84.mean,UV_A.mean,UV_B.mean,CS.mean
0,-1.152118,-1.165724,-1.184786,-1.164180,1.183900,1.450265,1.452978,1.434997,1.457705,1.461320,...,-0.505589,0.366171,1.391121,1.403482,1.386056,1.388303,1.395726,1.368135,1.646141,0.465797
1,-1.007757,-0.985763,-1.084375,-0.971958,1.334522,-0.379931,-0.384677,-0.371206,-0.384629,-0.383960,...,-0.437269,0.399029,0.708136,0.700791,0.711215,0.721715,0.710686,1.134346,1.007666,1.405871
2,0.660018,0.661762,0.650660,0.661274,-1.129266,-0.773445,-0.758459,-0.785488,-0.767050,-0.752301,...,2.576491,0.057558,-1.104416,-1.085595,-1.109935,-1.114246,-1.092305,-1.041048,-1.101192,2.119101
3,1.307588,1.310712,1.277200,1.311314,0.179504,-1.096252,-1.101805,-1.084556,-1.103892,-1.049062,...,-0.358749,0.274460,-1.480564,-1.486390,-1.498834,-1.509478,-1.478549,0.055543,-0.502242,-0.943734
4,-1.053919,-1.038391,-1.036889,-1.038849,0.801361,0.516960,0.508895,0.527165,0.503998,0.503094,...,-0.480464,0.619673,0.798102,0.788815,0.801032,0.802768,0.788059,1.107340,0.981379,-0.444076
